
# SHAP Phase 4 Analysis — freMTPL Claim Frequency Modelling

--- 

## Projektübersicht

Dieses Notebook dokumentiert die Datei `shap_phase4_analysis.py` und erklärt den vollständigen Workflow eines modernen actuarial Machine Learning Pipelines für die Schadenfrequenz-Modellierung auf Basis des **freMTPL-Datensatzes**.

Der Fokus liegt auf:

- Datenaufbereitung
- Aufbau eines klassischen GLM-Baselinesystems
- Training eines XGBoost-Poisson-Modells
- SHAP Explainability
- Vergleich zwischen GLM und Machine Learning
- Erstellung regulatorischer Reports
- Unit-Tests und Validierung

---

## Ziel des Projekts

Das Projekt demonstriert, wie moderne Explainable AI (XAI)-Methoden im Versicherungsumfeld eingesetzt werden können, um:

- nichtlineare Zusammenhänge zu modellieren,
- Modellentscheidungen nachvollziehbar zu machen,
- regulatorische Anforderungen (IFRS 17 / Solvency II) zu unterstützen,
- klassische aktuarielle Modelle mit Machine Learning zu vergleichen.




# 1. Importierte Bibliotheken

Die Datei verwendet ein professionelles Python-Setup für Data Science, Explainable AI und actuarial modelling.

### Verwendete Technologien

| Bibliothek | Zweck |
|---|---|
| pandas | Datenmanipulation |
| numpy | Numerische Berechnungen |
| matplotlib / seaborn | Visualisierung |
| xgboost | Gradient Boosting Modell |
| shap | Explainable AI |
| sklearn | ML-Utilities & GLM |
| logging | Produktionsfähiges Logging |



In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb



# 2. Architektur des Programms

Die Pipeline ist modular aufgebaut und folgt einer klaren Produktionsstruktur.

## Workflow

```text
1. Daten laden
2. Feature Engineering
3. Train/Test Split
4. GLM Baseline trainieren
5. XGBoost Modell trainieren
6. SHAP Werte berechnen
7. Visualisierungen erzeugen
8. Regulatorischen Report schreiben
9. Unit Tests ausführen
```

Die Struktur orientiert sich an produktionsreifen actuarial ML-Systemen.



# 3. Datenvorbereitung

Die Funktion `load_and_prepare()` übernimmt:

- Laden der CSV-Datei
- Schema-Validierung
- Exposure-Bereinigung
- Zielgrößenberechnung
- Feature Engineering

## Wichtige Features

| Feature | Beschreibung |
|---|---|
| Exposure | Versicherungsdauer |
| ClaimNb | Anzahl Schäden |
| ClaimFreq | Schadenfrequenz |
| LogDensity | logarithmierte Bevölkerungsdichte |
| BonusMalusCapped | begrenzter Bonus-Malus-Faktor |



In [ ]:

NUMERIC_FEATURES = [
    "VehPower",
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "Density"
]

CATEGORIC_FEATURES = [
    "VehBrand",
    "VehGas",
    "Area",
    "Region"
]



# 4. Encoding der kategorialen Variablen

Da XGBoost numerische Eingaben benötigt, werden kategoriale Variablen mit `LabelEncoder` transformiert.

## Vorteile

- einfache Implementierung
- geringe Speicherlast
- kompatibel mit Tree-Based Models

## Hinweis

Für produktive GLM-Systeme wäre häufig One-Hot-Encoding sinnvoller.



# 5. Poisson-GLM als Baseline

Das Projekt verwendet einen klassischen **Poisson Generalized Linear Model (GLM)** als Referenzmodell.

## Warum GLM?

GLMs sind im Versicherungswesen Standard, da sie:

- interpretierbar,
- regulatorisch akzeptiert,
- statistisch robust sind.

Das Modell verwendet:

- Poisson-Verteilung
- Log-Link-Funktion
- Exposure als Sample Weight



In [ ]:

from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler



# 6. XGBoost Poisson Modell

Das Herzstück des Projekts ist ein XGBoost-Regressor mit Poisson-Zielfunktion.

## Verwendete Parameter

| Parameter | Wert |
|---|---|
| objective | count:poisson |
| max_depth | 4 |
| learning_rate | 0.05 |
| n_estimators | 300 |
| subsample | 0.8 |

## Vorteile von XGBoost

- Modelliert Nichtlinearitäten
- Erkennt Interaktionen
- Sehr hohe Prognosequalität
- Robust gegenüber Ausreißern



In [ ]:

XGB_PARAMS = dict(
    objective="count:poisson",
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8
)



# 7. SHAP Explainability

SHAP (SHapley Additive exPlanations) erklärt die Vorhersagen des Modells auf Basis spieltheoretischer Konzepte.

## Ziel der SHAP-Analyse

- globale Feature Importance
- lokale Vorhersageerklärung
- regulatorische Transparenz
- Vergleich mit GLM-Effekten

## Verwendete SHAP-Visualisierungen

| Plot | Zweck |
|---|---|
| Beeswarm | globale Verteilung |
| Bar Plot | globale Wichtigkeit |
| Dependence Plot | Feature-Effekt |
| Waterfall Plot | individuelle Vorhersage |




# 8. Feature Importance

Das Projekt vergleicht zwei Arten von Wichtigkeit:

## XGBoost Gain

Misst:
- wie häufig ein Feature Splits verbessert

## Mean Absolute SHAP

Misst:
- tatsächlichen Einfluss auf Vorhersagen

Dieser Vergleich ist besonders wichtig für Explainability und regulatorische Reviews.



# 9. GLM vs SHAP Vergleich

Eine zentrale Stärke des Projekts ist der direkte Vergleich zwischen:

- linearem GLM
- nichtlinearem XGBoost

## Ziel

Überprüfung, ob die Haupteffekte des ML-Modells mit actuarial Erwartungen übereinstimmen.

## Beispiel

| Feature | GLM | SHAP |
|---|---|---|
| BonusMalus | positiv | positiv |
| DrivAge | negativ | negativ |

Abweichungen können auf:
- Nichtlinearitäten,
- Interaktionen,
- Multikollinearität
hinweisen.



# 10. Regulatorischer Report

Die Funktion `write_regulatory_report()` erstellt automatisch einen strukturierten Explainability-Report.

## Inhalt

- Modellbeschreibung
- wichtigste Features
- SHAP-Ergebnisse
- Vergleich GLM vs ML
- Interpretationen
- bekannte Limitationen

## Bedeutung

Dieser Teil simuliert reale regulatorische Dokumentation im Versicherungsumfeld.



# 11. Unit Tests

Das Projekt enthält produktionsnahe Smoke Tests.

## Validierte Bereiche

- Datenintegrität
- Modellvorhersagen
- SHAP Additivity
- verwendete Features

## Vorteil

Automatische Qualitätssicherung vor produktivem Einsatz.



# 12. Output-Dateien

Das Skript erzeugt mehrere Analyseartefakte:

```text
01_beeswarm_summary.png
02_bar_summary.png
03_dependence_<feature>.png
04_waterfall_case_<n>.png
05_gain_vs_shap_importance.png
06_shap_vs_glm_comparison.png
regulatory_report.txt
```

Diese Outputs eignen sich direkt für:

- wissenschaftliche Arbeiten
- GitHub-Projekte
- regulatorische Dokumentation
- Präsentationen



# 13. Technische Bewertung

## Stärken des Projekts

&#10003; Produktionsreife Struktur  
&#10003; Explainable AI integriert  
&#10003; Regulatorische Orientierung  
&#10003; Kombination aus GLM und ML  
&#10003; Unit Testing vorhanden  
&#10003; Saubere Logging-Architektur  

## Verbesserungspotenziale

- Hyperparameter-Tuning erweitern
- Cross Validation integrieren
- One-Hot-Encoding für GLM
- Model Registry ergänzen
- CI/CD Pipeline hinzufügen




# 14. Fazit

Das Projekt stellt eine moderne actuarial Machine Learning Pipeline dar, die klassische Versicherungsmodellierung mit Explainable AI kombiniert.

Besonders hervorzuheben sind:

- die professionelle Struktur,
- regulatorische Nachvollziehbarkeit,
- die SHAP-Integration,
- der Vergleich mit klassischen GLMs.

Damit eignet sich das Projekt hervorragend für:

- GitHub-Portfolios
- Masterarbeiten
- actuarial Data Science
- Explainable AI Demonstrationen
- regulatorische ML-Projekte

------------------------------------------------------------------------

# Kontakt

**Autor:** <marksquant@gmail.com>  
**Projekt:** Tarifmodellierung: GLM vs. Machine Learning; Analysis - freMTPL Claim Frequency Modelling      
**Sprache:** Python  
**Jahr:** 2026
